## Part A

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/alifhafedz/housing_price/refs/heads/main/housing_price_dataset.csv')  # Replace with your file path
print("Initial Data Shape:", df.shape)
print(df.head())

Initial Data Shape: (50000, 6)
   SquareFeet  Bedrooms  Bathrooms Neighborhood  YearBuilt          Price
0        2126         4          1        Rural       1969  215355.283618
1        2459         3          2        Rural       1980  195014.221626
2        1860         2          1       Suburb       1970  306891.012076
3        2294         2          1        Urban       1996  206786.787153
4        2130         5          2       Suburb       2001  272436.239065


##Data Cleaning

In [ ]:
for col in df.select_dtypes(include=np.number).columns:
    df[col].fillna(df[col].mean(), inplace=True)

# Fill missing categorical values with mode
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Remove duplicates
df.drop_duplicates(inplace=True)

# Remove outliers using Z-score for numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns
df = df[(np.abs(stats.zscore(df[numeric_cols])) < 3).all(axis=1)]

print("After Cleaning Shape:", df.shape)

After Cleaning Shape: (49965, 6)


/tmp/ipykernel_815/128284543.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].mean(), inplace=True)
/tmp/ipykernel_815/128284543.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'd

##Data Transformation

In [ ]:
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

# Encode categorical columns
for col in df.select_dtypes(include='object').columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

# Optional: Feature Engineering
# Example: create a 'TotalRooms' feature
if 'Bedrooms' in df.columns and 'Bathrooms' in df.columns:
    df['TotalRooms'] = df['Bedrooms'] + df['Bathrooms']

print("After Transformation Head:")
print(df.head())

After Transformation Head:
   SquareFeet  Bedrooms  Bathrooms  Neighborhood  YearBuilt     Price  \
0    0.207895  0.449282  -1.220097             0  -0.791876 -0.124972   
1    0.786793 -0.446574   0.005568             0  -0.260950 -0.392963   
2   -0.254528 -1.342431  -1.220097             1  -0.743610  1.080998   
3    0.499952 -1.342431  -1.220097             2   0.511307 -0.237861   
4    0.214849  1.345138   0.005568             1   0.752638  0.627062   

   TotalRooms  
0   -0.770815  
1   -0.441006  
2   -2.562528  
3   -2.562528  
4    1.350707  


##Data Reduction

In [ ]:
pca_cols = numeric_cols.tolist()
pca = PCA(n_components=min(5, len(pca_cols)))  # reduce to 5 principal components or fewer
principal_components = pca.fit_transform(df[pca_cols])

# Create a DataFrame with PCA components
pca_df = pd.DataFrame(principal_components, columns=[f'PC{i+1}' for i in range(principal_components.shape[1])])

# Combine PCA with any categorical features if needed
categorical_cols = df.select_dtypes(include='int64').columns.difference(numeric_cols)
final_df = pd.concat([pca_df, df[categorical_cols].reset_index(drop=True)], axis=1)

print("Final preprocessed Data Shape:", final_df.shape)
print(final_df.head())

Final preprocessed Data Shape: (49965, 6)
        PC1       PC2       PC3       PC4       PC5  Neighborhood
0  0.058670 -0.848250 -0.578726 -1.124368 -0.233025             0
1  0.246580 -0.457687 -0.042936  0.357725 -0.800579             0
2  0.468554 -1.923677  0.071855  0.072758  1.070757             1
3  0.065397 -1.466861  1.245726  0.003038 -0.388488             2
4  0.682484  1.158220  0.213002 -0.960605  0.199668             1


In [ ]:
final_df.to_csv('preprocessed_housing_data.csv', index=False)
print("Preprocessed data saved as 'preprocessed_housing_data.csv'")

Preprocessed data saved as 'preprocessed_housing_data.csv'


## Discussion

Several preprocessing techniques were applied to improve the quality of the dataset before machine learning model development. Missing values were handled using mean imputation for numerical attributes and mode imputation for categorical attributes. Duplicate records and outliers were also removed to reduce noise and improve data reliability.

Data transformation was performed through feature scaling and label encoding to ensure that the machine learning algorithms could process the features effectively. In addition, a new feature, TotalRooms, was created to provide more meaningful information from the existing attributes.

Finally, PCA was applied to reduce dimensionality while retaining most of the important information in the dataset. This helped reduce redundancy among features and improve computational efficiency during model training.

## Part B

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
import numpy as np

df = pd.read_csv('housing_price_dataset.csv')
df.head()

,SquareFeet,Bedrooms,Bathrooms,Neighborhood,YearBuilt,Price
0,2126,4,1,Rural,1969,215355.283618
1,2459,3,2,Rural,1980,195014.221626
2,1860,2,1,Suburb,1970,306891.012076
3,2294,2,1,Urban,1996,206786.787153
4,2130,5,2,Suburb,2001,272436.239065


## Data preparation

In [2]:
X = df.drop('Price', axis=1)
y = df['Price']

categorical = ['Neighborhood']
numeric = ['SquareFeet','Bedrooms','Bathrooms','YearBuilt']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## Train and evaluate 5 models

In [3]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'kNN': KNeighborsRegressor(),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

for name, model in models.items():

    pipe = Pipeline([
        ('prep', preprocessor),
        ('model', model)
    ])

    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_test)

    r2 = r2_score(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))

    print(f"\n{name}")
    print(f"R² Score: {r2:.4f}")
    print(f"RMSE: {rmse:.2f}")


Linear Regression
R² Score: 0.5756
RMSE: 49358.38

Decision Tree
R² Score: 0.0911
RMSE: 72230.46

Random Forest
R² Score: 0.5193
RMSE: 52529.06

kNN
R² Score: 0.4853
RMSE: 54353.53

Gradient Boosting
R² Score: 0.5742
RMSE: 49436.70


## Hyperparameter Tuning

In [4]:
from sklearn.model_selection import GridSearchCV

tuning_models = {
    'Decision Tree': (
        DecisionTreeRegressor(random_state=42),
        {
            'model__max_depth': [5,10,15]
        }
    ),

    'Random Forest': (
        RandomForestRegressor(random_state=42),
        {
            'model__n_estimators': [50,100],
            'model__max_depth': [10,20]
        }
    ),

    'kNN': (
        KNeighborsRegressor(),
        {
            'model__n_neighbors': [3,5,7]
        }
    ),

    'Gradient Boosting': (
        GradientBoostingRegressor(random_state=42),
        {
            'model__n_estimators': [50,100],
            'model__learning_rate': [0.05,0.1]
        }
    )
}

best_models = {}

for name, (model, params) in tuning_models.items():

    pipe = Pipeline([
        ('prep', preprocessor),
        ('model', model)
    ])

    grid = GridSearchCV(
        pipe,
        params,
        cv=3,
        scoring='r2',
        n_jobs=-1
    )

    grid.fit(X_train, y_train)

    best_models[name] = grid.best_estimator_

    print(f"\n{name}")
    print("Best Parameters:", grid.best_params_)


Decision Tree
Best Parameters: {'model__max_depth': 5}

Random Forest
Best Parameters: {'model__max_depth': 10, 'model__n_estimators': 100}

kNN
Best Parameters: {'model__n_neighbors': 7}

Gradient Boosting
Best Parameters: {'model__learning_rate': 0.05, 'model__n_estimators': 100}


## Performance Comparison

In [5]:
results = []

# Linear Regression
linear_model = Pipeline([
    ('prep', preprocessor),
    ('model', LinearRegression())
])

linear_model.fit(X_train, y_train)

pred = linear_model.predict(X_test)

results.append([
    'Linear Regression',
    r2_score(y_test, pred),
    np.sqrt(mean_squared_error(y_test, pred))
])

# Tuned models
for name, model in best_models.items():

    pred = model.predict(X_test)

    results.append([
        name,
        r2_score(y_test, pred),
        np.sqrt(mean_squared_error(y_test, pred))
    ])

results_df = pd.DataFrame(
    results,
    columns=['Model', 'R2 Score', 'RMSE']
)

results_df.sort_values(
    by='R2 Score',
    ascending=False
)

,Model,R2 Score,RMSE
0,Linear Regression,0.575563,49358.376911
4,Gradient Boosting,0.575198,49379.575831
1,Decision Tree,0.571378,49601.124955
2,Random Forest,0.567909,49801.421391
3,kNN,0.514388,52795.724905


## Discussion

The performance of the five machine learning models was compared using R² Score and RMSE. A higher R² Score indicates that the model explains a larger proportion of the variance in housing prices, while a lower RMSE indicates more accurate predictions.

Based on the results obtained, the model with the highest R² Score and the lowest RMSE was considered the best-performing model for housing price prediction. This model demonstrated better predictive capability compared to the other algorithms tested.